# Covariate balance check

Before trusting any ATE estimate, confirm that randomization actually held: the treatment and control groups should look statistically similar across all 12 features (`f0`-`f11`), since `treatment` was randomly assigned independent of any user trait.

Metric: **standardized mean difference (SMD)** per feature, `(mean_treated - mean_control) / pooled_std`. Convention: `|SMD| < 0.1` is considered well balanced.

In [1]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample
from uplift.balance import smd

df = load_sample()
df.shape

(1000000, 16)

In [2]:
df["treatment"].value_counts(normalize=True)

treatment
1    0.850849
0    0.149151
Name: proportion, dtype: float64

In [3]:
import pandas as pd

pd.crosstab(df["treatment"], df["exposure"])

exposure,0,1
treatment,,
0,149151,0
1,820285,30564


Confirms the logical constraint: `treatment=0` implies `exposure=0` — nobody blocked from targeting was ever shown an ad.

In [4]:
balance = smd(df).sort_values(key=abs, ascending=False)
balance

f3    -0.049987
f6    -0.041012
f5    -0.028587
f9     0.028086
f1     0.026698
f8    -0.026635
f7     0.019850
f10    0.015940
f4     0.015165
f11   -0.010457
f0    -0.007395
f2    -0.006078
dtype: float64

## Conclusion

All 12 SMDs are under the conventional 0.1 threshold (largest magnitude ~0.05), with no consistent directional pattern across features.

**But that threshold is too lenient to say randomization "held" without qualification.** Under true randomization an SMD has a standard error of roughly `sqrt(1/n_treated + 1/n_control)`, which is ~0.0028 on this 1M sample - so the largest SMD (~0.05) is already ~18 standard errors from zero. Notebook 07 finds the same imbalance on the full ~14M rows (~65 standard errors; the 12 features predict `treatment` with AUC ~0.51) and checks whether it moves the ATE.

Working conclusion: assignment is very nearly random, with a small but real dependence on the features. Safe to move on to ATE estimation using `treatment` as the group variable, carrying that caveat forward.